# 🏗️ Tamazight: Low-Resource LLM Fine-Tuning for the Amazigh Language (Tifinagh Script)

**Objective:** Fine-tune `Qwen/Qwen2.5-0.5B-Instruct` on Amazigh (Tifinagh) text using **QLoRA** (4-bit quantization), then export to **GGUF** for local inference via **Ollama**.

**Methodology:** QLoRA (Dettmers et al., 2023) allows us to fine-tune a quantized 4-bit model by training only low-rank adapter matrices, dramatically reducing GPU memory requirements while preserving model quality. This makes it feasible to fine-tune on a free-tier Google Colab T4 GPU (16 GB VRAM).

---

## 1 · Environment & Hardware Setup

We install the core Hugging Face ecosystem alongside `bitsandbytes` for 4-bit quantization support.
*All packages are pinned to recent stable releases compatible with the Colab T4 runtime.*


In [ ]:
# ── 1a: Install required packages ──
!pip install -q --upgrade \
    transformers>=4.41 \
    peft>=0.11 \
    trl>=0.9 \
    accelerate>=0.31 \
    bitsandbytes>=0.43 \
    datasets>=2.19


In [ ]:
# ── 1b: Verify GPU assignment ──
!nvidia-smi
import torch
print(f"\nPyTorch sees CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


---
## 2 · Data Acquisition & Bulletproof Formatting

### Why Tifinagh-only filtering?
Many multilingual corpora contain mixed scripts (Latin transliterations, Arabic, French loanwords).
For causal language modelling on a *single* script, we must enforce strict Unicode-range filtering
to prevent the model from learning spurious cross-script associations.
The **Tifinagh Unicode block** occupies **U+2D30 – U+2D7F** (80 code points).


In [ ]:
import re
from datasets import Dataset, DatasetDict, load_dataset

# ── 2a: Load raw data (with robust fallback) ──
RAW_SENTENCES = []
try:
    ds = load_dataset("abdelhaqueidali/Amazigh-English-Tatoeba-Extended", split="train")
    # The dataset may have varying column names; detect the Amazigh text column
    col = [c for c in ds.column_names if c.lower() in ("amazigh", "source", "text")]
    col = col[0] if col else ds.column_names[0]
    RAW_SENTENCES = [row[col] for row in ds if isinstance(row[col], str)]
    print(f"✅ Loaded {len(RAW_SENTENCES)} rows from HuggingFace.")
except Exception as e:
    print(f"⚠️  HuggingFace load failed ({e}). Using fallback data.")
    RAW_SENTENCES = [
        "ⴰⵣⵓⵍ ⴼⵍⵍⴰⵡⵏ, ⵎⴰⵏⵉⴽ ⴰⵜⵜⵉⵍⵉⵜ?",
        "ⵜⴰⵎⴰⵣⵉⵖⵜ ⴷ ⵜⵓⵜⵍⴰⵢⵜ ⵏⵏⵖ.",
        "ⴰⵔ ⵏⵜⵜⵉⵍⵉ ⴳ ⵜⵎⴰⵣⵉⵔⵜ.",
        "ⵉⵙⵎ ⵉⵏⵓ ⴰⵎⴰⵣⵉⵖ.",
        "ⵜⴰⵡⵊⴰ ⵏⵏⵖ ⵜⵍⵍⴰ ⴳ ⵓⴳⴰⴷⵉⵔ.",
        "ⵉⵎⴰⵍ ⵏⵏⵖ ⴳ ⵜⵎⵓⵔⵜ ⴰⴷ ⵉⴼⵓⵍⴽⵉ.",
        "ⴰⵙⵉⴼ ⵏ ⴷⵔⴰ ⵉⵖⵓⴷⴰ ⴱⴰⵀⵔⴰ.",
        "ⴰⵢⵜ ⵓⵎⴰⵍⵓ ⴳⴰⵏ ⴰⵢⵜ ⵜⵡⵉⵣⵉ.",
        "ⵜⴰⴼⵓⴽⵜ ⵜⵍⵍⴰ ⴳ ⵉⴳⵏⵏⴰ.",
        "ⴰⴷⵔⴰⵔ ⵏ ⵜⵓⴱⵇⴰⵍ ⵉⵖⵓⴷⴰ.",
    ]

print(f"Raw sentences count: {len(RAW_SENTENCES)}")


In [ ]:
# ── 2b: Tifinagh-only filter ──
# Keep ONLY characters in the Tifinagh Unicode block (U+2D30-U+2D7F),
# whitespace, and basic punctuation marks.
TIFINAGH_RE = re.compile(r"[^\u2D30-\u2D7F\s\.\,\!\?]")

def clean_tifinagh(text: str) -> str:
    """Remove any character outside the Tifinagh Unicode block and basic punctuation."""
    cleaned = TIFINAGH_RE.sub("", text)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

cleaned = [clean_tifinagh(s) for s in RAW_SENTENCES]
cleaned = [s for s in cleaned if len(s) >= 4]  # discard very short fragments
print(f"After Tifinagh filtering: {len(cleaned)} sentences retained.")
print("Sample:", cleaned[:3])


In [ ]:
# ── 2c: Build HuggingFace Dataset with single 'text' column ──
# CRITICAL: SFTTrainer expects a Dataset with a 'text' column for causal LM.
# We do NOT use chat templates — each row is a raw cleaned Tifinagh string.
full_ds = Dataset.from_dict({"text": cleaned})
split = full_ds.train_test_split(test_size=0.05, seed=42)
dataset = DatasetDict({"train": split["train"], "test": split["test"]})
print(dataset)
print("\nFirst training example:", dataset["train"][0])


---
## 3 · QLoRA Configuration & Model Loading

### Why QLoRA?
**QLoRA** (Dettmers et al., 2023) enables fine-tuning of large language models on consumer GPUs by:
1. **4-bit NormalFloat quantization** — reduces the memory footprint of frozen base weights by ~4×.
2. **Double quantization** — quantizes the quantization constants themselves, saving ~0.4 bits/param.
3. **Paged optimizers** — offloads optimizer states to CPU RAM on OOM, critical for the 16 GB T4.

We keep the base weights frozen in 4-bit and only train low-rank adapters (LoRA) in `bfloat16`.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# ── 3a: 4-bit quantization config ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                # NormalFloat4 — optimal for normally-distributed weights
    bnb_4bit_compute_dtype=torch.bfloat16,     # compute in bf16 for numerical stability
    bnb_4bit_use_double_quant=True,            # double quantization for extra memory savings
)

# ── 3b: Load model ──
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# ── 3c: Load tokenizer ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Qwen uses eos as pad by convention
print(f"Model loaded. dtype={model.dtype}, device={model.device}")


---
## 4 · LoRA Adapter Setup

### Target Module Selection Rationale
We target **all linear projection layers** in each transformer block:

| Module | Role |
|--------|------|
| `q_proj`, `k_proj`, `v_proj`, `o_proj` | Self-attention projections |
| `gate_proj`, `up_proj`, `down_proj` | Feed-forward network (SwiGLU MLP) |

By adapting both attention *and* MLP layers, the model gains sufficient capacity to learn
a new script's token co-occurrence patterns — essential for low-resource script adaptation.

**Rank (r=16):** A moderate rank provides enough capacity for script-level adaptation without
overfitting on a small corpus. The effective learning rate is scaled by `alpha/r = 32/16 = 2`.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── 4a: Prepare model for QLoRA training ──
# This freezes the base model, casts LayerNorm to fp32, and enables gradient checkpointing.
model = prepare_model_for_kbit_training(model)

# ── 4b: LoRA configuration ──
lora_config = LoraConfig(
    r=16,                              # rank — controls adapter expressiveness
    lora_alpha=32,                     # scaling factor (effective lr multiplier = alpha/r = 2)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention projections
        "gate_proj", "up_proj", "down_proj",       # MLP projections (SwiGLU)
    ],
    lora_dropout=0.05,                 # light regularisation to prevent overfitting
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# ── 4c: Print trainable parameter summary ──
model.print_trainable_parameters()


---
## 5 · Training Execution

We use the `SFTTrainer` (Supervised Fine-Tuning Trainer) from TRL, which wraps the
Hugging Face `Trainer` with convenience features for language-model fine-tuning.

**Hyperparameter choices:**
- **Effective batch size** = `per_device_train_batch_size` × `gradient_accumulation_steps` = 4 × 4 = **16**
- **Learning rate 2e-4** with cosine decay — standard for QLoRA (Dettmers et al., 2023)
- **Paged AdamW 8-bit** — memory-efficient optimizer that offloads pages to CPU on OOM
- **`max_steps=300`** — sufficient for a small low-resource corpus to converge without overfitting
- **`bf16=True`** — bfloat16 mixed precision, native to the T4's Tensor Cores


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    max_steps=300,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=30,
    logging_steps=25,
    save_steps=100,
    optim="paged_adamw_8bit",
    bf16=True,
    report_to="none",                  # disable W&B / MLflow logging for simplicity
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",          # CRITICAL: column name in our Dataset
    max_seq_length=256,                 # Tifinagh sentences are typically short
    args=training_args,
    peft_config=None,                   # already applied via get_peft_model above
)

print("🚀 Starting training...")
trainer.train()
print("✅ Training complete.")


---
## 6 · Quick Inference Test

Before exporting, we verify the fine-tuned adapter produces coherent Tifinagh continuations.
We flush GPU cache first to reclaim memory used by training-time activation tensors.


In [ ]:
import torch

# ── 6a: Flush GPU memory ──
torch.cuda.empty_cache()

# ── 6b: Inference function ──
def generate_tifinagh(prompt: str, max_new_tokens: int = 20):
    """Generate a continuation of a Tifinagh prompt using the fine-tuned LoRA model."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
        )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt : {prompt}")
    print(f"Output : {result}")
    return result

# ── 6c: Test with a Tifinagh greeting ──
generate_tifinagh("ⴰⵣⵓⵍ ⴼ")


---
## 7 · Merge & GGUF Export Pipeline

### Export Strategy
1. **Merge** LoRA adapters back into the base model weights (float16).
2. **Convert** the merged Hugging Face model to GGUF format using `llama.cpp`'s converter.
3. **Quantize** the F16 GGUF down to `Q4_K_M` — a 4-bit quantization scheme that retains
   excellent quality while enabling fast CPU inference locally.


In [ ]:
import torch, gc

# ── 7a: Flush GPU memory ──
torch.cuda.empty_cache()
gc.collect()

# ── 7b: Merge LoRA weights into base model ──
print("Merging LoRA adapters into base model (float16)...")
merged_model = model.merge_and_unload()

MERGED_DIR = "./merged_qwen_amazigh"
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"✅ Merged model saved to {MERGED_DIR}")


In [ ]:
# ── 7c: Clone llama.cpp and compile ──
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
%cd llama.cpp
!make -j$(nproc)
%cd ..
print("✅ llama.cpp compiled.")


In [ ]:
# ── 7d: Convert HF model → F16 GGUF ──
!pip install -q gguf sentencepiece
!python llama.cpp/convert_hf_to_gguf.py ./merged_qwen_amazigh \
    --outfile amazigh-qwen-f16.gguf --outtype f16
print("✅ F16 GGUF created.")


In [ ]:
# ── 7e: Quantize F16 → Q4_K_M ──
!./llama.cpp/llama-quantize amazigh-qwen-f16.gguf amazigh-qwen-q4.gguf Q4_K_M
print("✅ Q4_K_M GGUF created: amazigh-qwen-q4.gguf")


---
## 8 · Ollama Packaging & Local Download

We create an **Ollama `Modelfile`** that wraps the quantized GGUF with the **ChatML** template
used by the Qwen2.5 family, then package everything into a single downloadable zip archive.


In [ ]:
# ── 8a: Generate Modelfile ──
# The Modelfile tells Ollama how to load our GGUF and which chat template to use.
modelfile_lines = [
    "FROM ./amazigh-qwen-q4.gguf",
    'TEMPLATE """',
    "{{- if .System }}<|im_start|>system",
    "{{ .System }}<|im_end|>",
    "{{- end }}",
    "{{- range .Messages }}<|im_start|>{{ .Role }}",
    "{{ .Content }}<|im_end|>",
    "{{- end }}<|im_start|>assistant",
    '"""',
    'PARAMETER stop "<|im_start|>"',
    'PARAMETER stop "<|im_end|>"',
]

modelfile_content = "\n".join(modelfile_lines) + "\n"

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

print("✅ Modelfile written. Contents:")
print("-" * 50)
print(open("Modelfile").read())
print("-" * 50)


In [ ]:
# ── 8b: Zip GGUF + Modelfile for download ──
import shutil, os

EXPORT_DIR = "./ollama_export"
os.makedirs(EXPORT_DIR, exist_ok=True)
shutil.copy("amazigh-qwen-q4.gguf", EXPORT_DIR)
shutil.copy("Modelfile", EXPORT_DIR)

# Create the zip archive
shutil.make_archive("ollama_amazigh_model", "zip", EXPORT_DIR)
print("✅ ollama_amazigh_model.zip created.")
print(f"   Size: {os.path.getsize('ollama_amazigh_model.zip') / 1e6:.1f} MB")


In [ ]:
# ── 8c: Trigger browser download (Colab only) ──
from google.colab import files
files.download('ollama_amazigh_model.zip')


---
## 🖥️ Local Deployment Instructions (Windows / macOS / Linux)

After downloading and extracting `ollama_amazigh_model.zip`, open a terminal
**inside the extracted folder** and run the following two commands:

```bash
# Step 1: Create the Ollama model from the Modelfile
ollama create amazigh-qwen -f Modelfile

# Step 2: Run the model interactively
ollama run amazigh-qwen
```

> **Prerequisites:** [Ollama](https://ollama.com/) must be installed and running on your machine.
> On Windows, download the installer from the official site. On Linux/macOS, use:
> `curl -fsSL https://ollama.com/install.sh | sh`

---
*Notebook generated for the **Tamasight** project — Low-Resource LLM Benchmark for the Amazigh Language.*
